# 04 - Reaction Pathways and Constrained-Optimization Landscapes

The final notebook targets the most thesis-like part of the file:
reaction-pathway and constrained-optimization information.

The database contains many rows whose `Name` or `Path` include `copt`.
These are not static adsorption points, but snapshots along a constrained
optimization or pathway construction.

Here we convert that latent information into interpretable reaction profiles.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import subprocess
import sys
import tempfile

sys.path.insert(0, r"/Users/dk2994/Desktop/git/DFTDataFrame/src")
import DFTDataFrame.Tools as OP

pd = OP.pd
np = OP.np
plt = OP.plt

DATA_PATH = Path(r"/Users/dk2994/Desktop/Uni/scripts/created_frame.hdf")
DFTDATAFRAME_SRC = Path(r"/Users/dk2994/Desktop/git/DFTDataFrame/src")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def load_onepiece_hdf(path=DATA_PATH, key="df"):
    """Load the local OnePiece-style HDF table.

    This notebook uses the local DFTDataFrame package as the available
    OnePiece-compatible analysis layer. The actual HDF payload is read through
    the pandas namespace exposed by that package.
    """
    path = Path(path)
    try:
        return OP.pd.read_hdf(path, key=key).copy()
    except Exception as original_error:
        helper_python = Path("/Users/dk2994/opt/anaconda3/bin/python")
        if not helper_python.exists():
            raise original_error
        output = Path(tempfile.NamedTemporaryFile(delete=False, suffix=".pkl", prefix="created_frame_").name)
        script = """
from pathlib import Path
import sys
import numpy as np
import pandas as pd
try:
    import numpy.core as numpy_core
    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
    import ase.constraints  # noqa: F401
except Exception:
    pass
source = Path(sys.argv[1])
key = sys.argv[2]
target = Path(sys.argv[3])
pd.read_hdf(source, key=key).to_pickle(target)
"""
        completed = subprocess.run(
            [str(helper_python), "-c", script, str(path), key, str(output)],
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            detail = completed.stderr.strip() or completed.stdout.strip()
            raise RuntimeError(f"Could not read {path}. Helper reader failed: {detail}") from original_error
        return OP.pd.read_pickle(output).copy()


ADSORBATE_TOKENS = [
    "CH3OH", "CH3O", "HCOOH", "H2COOH", "HCOO", "COOH", "CO2", "HCO", "CO"
]


def guess_adsorbate(name):
    if not isinstance(name, str):
        return ""
    for token in sorted(ADSORBATE_TOKENS, key=len, reverse=True):
        if re.search(rf"(^|[-_%]){re.escape(token)}($|[-_%])", name):
            return token
    return ""


def guess_record_class(name, path=""):
    text = f"{name} {path}".lower()
    if "gasphases" in text:
        return "gas"
    if "copt" in text:
        return "copt"
    if "convergence" in text:
        return "convergence"
    if "bulk" in text:
        return "bulk"
    if "clean" in text:
        return "clean_surface"
    if guess_adsorbate(name):
        return "adsorbate"
    return "other"


def guess_facet(name):
    if not isinstance(name, str):
        return ""
    for facet in ["100", "110", "111", "211", "221"]:
        if facet in name:
            return facet
    return ""


def material_family(name):
    if not isinstance(name, str):
        return "unknown"
    token = name.split("-")[0]
    return token or "unknown"


def surface_key_from_name(name):
    if not isinstance(name, str):
        return ""
    key = name
    key = re.sub(r"-copt-.*$", "", key)
    key = re.sub(r"-(CH3OH|CH3O|HCOOH|H2COOH|HCOO|COOH|CO2|HCO|CO)([-_%].*)?$", "", key)
    key = re.sub(r"-[0-9]+$", "", key)
    return key


def add_taxonomy(df):
    out = df.copy()
    out["record_class"] = [guess_record_class(n, p) for n, p in zip(out["Name"], out["Path"], strict=False)]
    out["facet"] = out["Name"].map(guess_facet)
    out["material_family"] = out["Name"].map(material_family)
    out["adsorbate"] = out["Name"].map(guess_adsorbate)
    out["surface_key"] = out["Name"].map(surface_key_from_name)
    return out


def formula_counts(formula):
    counts = {}
    if not isinstance(formula, str):
        return counts
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


def add_formula_features(df):
    out = df.copy()
    parsed = out["Formula"].map(formula_counts)
    all_elements = sorted({el for counts in parsed for el in counts})
    for el in all_elements:
        out[el] = parsed.map(lambda counts: counts.get(el, 0))
    out["n_elements"] = parsed.map(len)
    out["n_atoms_formula"] = parsed.map(lambda counts: sum(counts.values()))
    return out


def gas_reference_energy(df, token):
    pattern = rf"^gasphases-{re.escape(token)}(?:$|[-_].*)"
    subset = df[df["Name"].astype(str).str.contains(pattern, regex=True, case=False, na=False)]
    subset = subset[pd.to_numeric(subset["E"], errors="coerce").notna()]
    if subset.empty:
        return np.nan
    return float(subset["E"].iloc[0])


def assign_clean_references(df):
    out = df.copy()
    clean = out[out["record_class"] == "clean_surface"].copy()
    clean = clean[pd.to_numeric(clean["E"], errors="coerce").notna()]
    clean_map = clean.groupby("surface_key")["E"].min().to_dict()
    clean_name_map = clean.sort_values("E").drop_duplicates("surface_key").set_index("surface_key")["Name"].to_dict()
    out["clean_ref_E"] = out["surface_key"].map(clean_map)
    out["clean_ref_name"] = out["surface_key"].map(clean_name_map)
    out["delta_E_surface"] = pd.to_numeric(out["E"], errors="coerce") - pd.to_numeric(out["clean_ref_E"], errors="coerce")
    return out


def adsorption_energy_conventions(df):
    out = df.copy()
    e_co = gas_reference_energy(out, "CO")
    e_h2 = gas_reference_energy(out, "H2")
    e_ch3oh = gas_reference_energy(out, "CH3OH")
    e_hcooh = gas_reference_energy(out, "HCOOH")
    out["E_ads_CO"] = np.where(out["adsorbate"].eq("CO"), out["E"] - out["clean_ref_E"] - e_co, np.nan)
    out["E_ads_CH3O_from_CH3OH"] = np.where(
        out["adsorbate"].eq("CH3O"),
        out["E"] - out["clean_ref_E"] - e_ch3oh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_HCOO_from_HCOOH"] = np.where(
        out["adsorbate"].eq("HCOO"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_COOH_from_HCOOH"] = np.where(
        out["adsorbate"].eq("COOH"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    return out


df_raw = load_onepiece_hdf()
df = add_formula_features(add_taxonomy(df_raw))
df["E"] = pd.to_numeric(df["E"], errors="coerce")
df["fmax"] = pd.to_numeric(df["fmax"], errors="coerce")
df["has_structure"] = df["struc"].map(lambda value: value.__class__.__name__ == "Atoms")
df["has_contcar"] = df["CONTCAR"].map(lambda value: value.__class__.__name__ == "Atoms")

## 1. Select constrained-optimization rows

In [ ]:
copt = df[df["record_class"].eq("copt")].copy()
print("number of copt rows:", len(copt))
copt[["Name", "Path", "Formula", "E", "fmax"]].head(20)

## 2. Parse pathway identities and image indices from the names

In [ ]:
def parse_copt_name(name):
    if not isinstance(name, str) or "copt" not in name:
        return pd.Series({"pathway": "", "image": np.nan, "state_pair": ""})
    image_match = re.search(r"-(\d{2})$", name)
    image = int(image_match.group(1)) if image_match else np.nan
    state_match = re.search(r"copt-([^%]+)%([^-]+(?:-[^-]+)*)", name)
    if state_match:
        state_pair = state_match.group(1) + " -> " + state_match.group(2)
    else:
        state_pair = ""
    series_key = re.sub(r"-\d{2}$", "", name)
    return pd.Series({"pathway": series_key, "image": image, "state_pair": state_pair})

copt = pd.concat([copt, copt["Name"].apply(parse_copt_name)], axis=1)
copt[["Name", "pathway", "image", "state_pair", "E"]].head(20)

## 3. Which pathway families are present most often?

In [ ]:
pathway_counts = copt["state_pair"].replace("", np.nan).dropna().value_counts().head(15)
pathway_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
pathway_counts.sort_values().plot(kind="barh", ax=ax, color="#b279a2")
ax.set_title("Most frequent constrained-optimization transformations")
ax.set_xlabel("rows")
ax.set_ylabel("state pair")
plt.tight_layout()
plt.show()

## 4. Build image-resolved energy profiles

In [ ]:
profiles = (
    copt.dropna(subset=["image", "E"])
    .groupby("pathway")
    .agg(n_images=("image", "size"), min_image=("image", "min"), max_image=("image", "max"))
    .query("n_images >= 5")
    .sort_values("n_images", ascending=False)
)
profiles.head(20)

In [ ]:
top_profiles = profiles.head(4).index.tolist()
fig, axes = plt.subplots(len(top_profiles), 1, figsize=(10, 3.4 * len(top_profiles)), sharex=False)
if len(top_profiles) == 1:
    axes = [axes]

for ax, pathway in zip(axes, top_profiles, strict=False):
    prof = copt[copt["pathway"].eq(pathway)].sort_values("image").copy()
    x = prof["image"].to_numpy()
    y = prof["E"].to_numpy()
    ax.plot(x, y, marker="o", color="#4c78a8")
    ax.set_title(pathway)
    ax.set_ylabel("E / eV")
    ax.set_xlabel("image index")
plt.tight_layout()
plt.show()

## 5. Smooth one profile with the OnePiece barrier helper

In [ ]:
example_pathway = top_profiles[0]
example = copt[copt["pathway"].eq(example_pathway)].sort_values("image").head(3).copy()
example[["Name", "image", "E"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = example["image"].to_numpy()
y = example["E"].to_numpy()
ax.scatter(x, y, color="black", zorder=3, label="calculated images")
if len(example) == 3:
    OP.barrier(float(x[0]), float(x[1]), float(x[2]), float(y[0]), float(y[1]), float(y[2]), color="crimson")
ax.set_title("Illustrative barrier interpolation with OP.barrier")
ax.set_xlabel("image index")
ax.set_ylabel("E / eV")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Chemical reading of the pathway space

At this point the HDF file can be read in a thesis-like way:

- static adsorption states provide thermodynamic anchors,
- clean surfaces provide reference energies,
- `copt` sequences provide mechanistic continuity between states,
- and the local `DFTDataFrame` tools already contain enough functionality to
  bridge tabular database work with atomistic interpretation.

That is exactly why a single, well-curated `.hdf` file can support a much
larger scientific narrative than a bare spreadsheet of energies.